# NutriDermAI — R-LLaVA Stage I + II Training (DGX, No Unsloth)
#
# Pure transformers + peft, 4-bit NF4, 3× A100-SXM4-40GB
# No unsloth — removed due to torch version conflicts with CUDA 12.2 driver
#
# Cell order:
#   1: Imports + Config + Dataset Download
#   2: HuggingFace Login
#   3: Load Model (4-bit NF4)
#   4: Image + Token Utilities
#   5: Stage I — Projector Pretraining (VICReg alignment)
#   6: Stage II Datasets + DataLoaders
#   7: Stage II — Apply LoRA + Train
#   8: Evaluation — BLEU-4, ROUGE-L, METEOR, Exact Match
#
# Fixes applied:
#   - weights_only=True for projector.pt (PyTorch 2.6)
#   - gradient_checkpointing_enable() before Stage II training
#   - RESUME_FROM restructured: check before get_peft_model
#   - get_projector() helper: robust multi_modal_projector lookup
#   - max_len raised to 4096 (was 2048, tight with 1752 image tokens)
#   - local_files_only=True in eval cell (no HF re-download)
#   - num_workers=4 (was 8) to reduce NFS file handle pressure


In [ ]:
# Cell 1: Imports + Config + Dataset Download
import os, sys, json, gc, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch.nn as nn

warnings.filterwarnings("ignore")
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert torch.cuda.is_available(), "No GPU — check environment"

n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    sm   = torch.cuda.get_device_capability(i)
    print(f"  GPU {i}: {name}  {vram:.1f} GB  SM {sm[0]}.{sm[1]}")

# Paths
WORK_DIR       = Path('/data/Stagewise Dataset/Stage7')
CHECKPOINT_DIR = WORK_DIR / 'rllava_checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Download dataset
print("\nDownloading dataset via kagglehub...")
import kagglehub
dataset_path = Path(kagglehub.dataset_download("akhilkulal/nutriderm-stage7-vqa"))
print(f"Dataset path: {dataset_path}")

TRAIN_PARQUET = dataset_path / 'stage7_vqa_train.parquet'
VAL_PARQUET   = dataset_path / 'stage7_vqa_val.parquet'
if not TRAIN_PARQUET.exists():
    for f in dataset_path.rglob('stage7_vqa_train.parquet'): TRAIN_PARQUET = f; break
if not VAL_PARQUET.exists():
    for f in dataset_path.rglob('stage7_vqa_val.parquet'):   VAL_PARQUET   = f; break
assert TRAIN_PARQUET.exists(), f"train parquet not found under {dataset_path}"
assert VAL_PARQUET.exists(),   f"val parquet not found under {dataset_path}"
print(f"Train: {TRAIN_PARQUET}\nVal  : {VAL_PARQUET}")

# Model
BASE_MODEL_ID  = 'llava-hf/llava-v1.6-mistral-7b-hf'
IMG_SIZE       = 336
GRID_PINPOINTS = [[336,672],[672,336],[672,672],[1008,336],[336,1008]]
ALPHA_MODE     = 'dynamic'

# Stage I
S1_SUBSAMPLE   = None    # None = all unique images
S1_LR          = 1e-3
S1_GRAD_ACCUM  = 4
S1_EPOCHS      = 3
SKIP_STAGE1    = False

# Stage II
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
MAX_TRAIN_SAMPLES   = None
MAX_VAL_SAMPLES     = None
STAGE2_LR           = 2e-5
STAGE2_BATCH        = 4
STAGE2_GRAD_ACCUM   = 4
STAGE2_EPOCHS       = 7
STAGE2_WARMUP       = 0.03
STAGE2_MAX_NEW_TOK  = 64
EARLY_STOP_PATIENCE = 5
# ↑ Raise max_len to give more room for answers alongside 1752 image tokens
MAX_SEQ_LEN         = 4096   # was 2048; Mistral-7B supports 32768

# Resume — checked BEFORE LoRA init in Cell 7
RESUME_FROM = None
resume_p = CHECKPOINT_DIR / 'rllava_stage2_best'
if resume_p.exists():
    RESUME_FROM = str(resume_p)
    print(f"Resume checkpoint found: {RESUME_FROM}")

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

train_df = pd.read_parquet(TRAIN_PARQUET)
val_df   = pd.read_parquet(VAL_PARQUET)
if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(MAX_TRAIN_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES and len(val_df) > MAX_VAL_SAMPLES:
    val_df = val_df.sample(MAX_VAL_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nTrain: {len(train_df):,}  Val: {len(val_df):,}")
if 'question_type' in train_df.columns:
    print(f"Q-types: {dict(train_df['question_type'].value_counts())}")


In [ ]:
# Cell 2: HuggingFace Login
HF_TOKEN = os.environ.get("HF_TOKEN", "")
# HF_TOKEN = "hf_xxxxxxxxxxxx"  # uncomment and paste your token

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub")
else:
    print("No HF_TOKEN — set os.environ['HF_TOKEN'] = 'hf_xxx' if download fails")


In [ ]:
# Cell 3: Load Model (4-bit, NO unsloth, pure transformers+peft)
gc.collect()
for i in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()

from transformers import (
    LlavaNextForConditionalGeneration,
    LlavaNextProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

sm = torch.cuda.get_device_capability(0)
print(f"GPU SM: {sm[0]}.{sm[1]}")

# 4-bit NF4 — works on A100 SM 8.0 with bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading LLaVA-v1.6-Mistral-7B (4-bit NF4)...")
model = LlavaNextForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
# Disable KV-cache during training (required for gradient checkpointing)
model.config.use_cache = False
# Enable gradient checkpointing — essential for 7B on 40GB GPUs
# Must be called before get_peft_model in Cell 7
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

processor = LlavaNextProcessor.from_pretrained(BASE_MODEL_ID)
text_tok  = processor.tokenizer
if text_tok.pad_token is None:
    text_tok.pad_token    = text_tok.eos_token
    text_tok.pad_token_id = text_tok.eos_token_id

for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    free = (torch.cuda.get_device_properties(i).total_memory
            - torch.cuda.memory_allocated(i)) / 1e9
    print(f"GPU {i}: {used:.2f} GB used | {free:.2f} GB free")
print("Model loaded (4-bit NF4, gradient checkpointing enabled).")


In [ ]:
# Cell 4: Image + Token Utilities
from transformers.models.llava_next.image_processing_llava_next import select_best_resolution

IMAGE_TOKEN_ID = 32000
_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073]).view(3,1,1)
_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1)

def encode_images(images):
    def _tile(img, h, w):
        a = np.array(img.resize((w,h)), dtype=np.float32).transpose(2,0,1) / 255.
        return (torch.from_numpy(a) - _MEAN) / _STD
    all_tiles, sizes = [], []
    for img in images:
        if not isinstance(img, Image.Image):
            img = Image.new('RGB', (IMG_SIZE,IMG_SIZE), (128,128,128))
        img = img.convert('RGB')
        bh, bw = select_best_resolution((img.height,img.width), GRID_PINPOINTS)
        tiles  = []; resized = img.resize((bw,bh))
        for r in range(0, bh, IMG_SIZE):
            for c in range(0, bw, IMG_SIZE):
                tiles.append(_tile(resized.crop((c,r,c+IMG_SIZE,r+IMG_SIZE)), IMG_SIZE, IMG_SIZE))
        tiles.append(_tile(img, IMG_SIZE, IMG_SIZE))
        all_tiles.append(tiles); sizes.append([bh, bw])
    max_t = max(len(t) for t in all_tiles)
    pv = torch.zeros(len(images), max_t, 3, IMG_SIZE, IMG_SIZE)
    for i, tiles in enumerate(all_tiles):
        for j, t in enumerate(tiles): pv[i,j] = t
    return pv, torch.tensor(sizes, dtype=torch.long)

# TOKENS_PER_PATCH = 584 confirmed (576 patch tokens + 8 image-newline tokens per tile)
try:
    vis_cfg = getattr(model.config, 'vision_config', None)
    ps  = getattr(vis_cfg, 'patch_size', 14) if vis_cfg else 14
    isz = getattr(vis_cfg, 'image_size', 336) if vis_cfg else 336
    TOKENS_PER_PATCH = (isz // ps) ** 2
    if TOKENS_PER_PATCH == 576: TOKENS_PER_PATCH = 584   # add 8 image-newline tokens
except Exception:
    TOKENS_PER_PATCH = 584
print(f"TOKENS_PER_PATCH = {TOKENS_PER_PATCH}")

try:
    from transformers.models.llava_next.modeling_llava_next import image_size_to_num_patches as _r
    def _n_patches(sz, gp, ps):
        s = sz[0] if (isinstance(sz,list) and isinstance(sz[0],list)) else sz
        return _r(s, gp, ps)
except ImportError:
    def _n_patches(sz, gp, ps):
        s = sz[0] if (isinstance(sz,list) and isinstance(sz[0],list)) else sz
        return (s[0]//ps) * (s[1]//ps) + 1

def build_input_ids(text, image_sizes_list, max_len=None):
    """
    Tokenise text, expand <image> placeholder to n_img image tokens.
    max_len defaults to MAX_SEQ_LEN (4096) — raised from 2048 to avoid
    silently truncating answers alongside 1752 image tokens.
    """
    if max_len is None:
        max_len = MAX_SEQ_LEN
    n_tiles = _n_patches(image_sizes_list, GRID_PINPOINTS, IMG_SIZE)
    n_img   = n_tiles * TOKENS_PER_PATCH
    raw     = text_tok.encode(text, add_special_tokens=True)
    img_pos = next((k for k,t in enumerate(raw) if t == IMAGE_TOKEN_ID), None)
    if img_pos is None:
        seq = text_tok.encode('<image>', add_special_tokens=False)
        for k in range(len(raw)-len(seq)+1):
            if raw[k:k+len(seq)] == seq:
                img_pos = k; raw = raw[:k]+[IMAGE_TOKEN_ID]+raw[k+len(seq):]; break
    if img_pos is None:
        pad = (raw + [text_tok.pad_token_id]*max_len)[:max_len]
        return pad, [1 if t != text_tok.pad_token_id else 0 for t in pad]
    expanded = raw[:img_pos] + [IMAGE_TOKEN_ID]*n_img + raw[img_pos+1:]
    if len(expanded) > max_len:
        avail    = max_len - n_img - img_pos
        expanded = raw[:img_pos] + [IMAGE_TOKEN_ID]*n_img + (raw[img_pos+1:][:avail] if avail>0 else [])
    pad = (expanded + [text_tok.pad_token_id]*max_len)[:max_len]
    return pad, [1 if t != text_tok.pad_token_id else 0 for t in pad]

def _mask_after_inst(ids):
    IE = text_tok.encode('[/INST]', add_special_tokens=False); labels = list(ids)
    for j in range(len(labels)-len(IE)+1):
        if labels[j:j+len(IE)] == IE:
            for k in range(j+len(IE)): labels[k] = -100
            break
    return labels

def _parse_bbox(raw):
    if isinstance(raw, (list, np.ndarray)): return [int(v) for v in raw]
    s = str(raw).replace('(','').replace(')','').replace('[','').replace(']','').strip()
    parts = [p.strip() for p in s.split(',') if p.strip()]
    if len(parts) != 4: parts = s.split()
    return [int(float(p)) for p in parts]

def free_memory():
    gc.collect()
    for i in range(torch.cuda.device_count()):
        torch.cuda.empty_cache()

def get_projector(m):
    """
    Robustly find multi_modal_projector regardless of PeftModel wrapping depth.
    Walks: m -> m.model -> m.model.model until the attribute is found.
    """
    for attr in ['multi_modal_projector']:
        node = m
        for _ in range(4):   # max 4 nesting levels
            if hasattr(node, attr):
                return getattr(node, attr)
            if hasattr(node, 'model'):
                node = node.model
            elif hasattr(node, 'base_model'):
                node = node.base_model
            else:
                break
    raise AttributeError(f"Could not find multi_modal_projector in model tree")

print("Utilities ready.")


In [ ]:
# Cell 5: Stage I — Projector Pretraining (bypasses Mistral for speed)
s1_ckpt = None
s1_log  = []
s1_best = float('inf')

if SKIP_STAGE1:
    print("Stage I SKIPPED. Using pretrained LLaVA-v1.6 projector.")
else:
    print("Stage I starting...")
    free_memory()

    for p in model.parameters():
        p.requires_grad = False

    import bitsandbytes as bnb
    proj = get_projector(model)

    def dequant_module(module):
        for name, param in list(module.named_parameters(recurse=False)):
            if isinstance(param, bnb.nn.Params4bit):
                fp16 = param.data.to(torch.float16)
                setattr(module, name, nn.Parameter(fp16, requires_grad=True))
            elif param.dtype not in (torch.float16, torch.bfloat16):
                fp16 = param.data.cpu().to(torch.float16).to(param.device)
                setattr(module, name, nn.Parameter(fp16, requires_grad=True))
            else:
                param.requires_grad = True
        for child in module.children():
            dequant_module(child)

    dequant_module(proj)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable: {n_train:,} params (projector only)")
    assert n_train > 0, "No trainable params in projector"

    vision_tower = (model.model.vision_tower if hasattr(model, 'model')
                    else model.vision_tower)  # always frozen
    projector = proj

    def get_clip_features(pixel_values):
        # CLIP encoder — always frozen, always no_grad
        B, n_tiles, C, H, W = pixel_values.shape
        pv_flat = pixel_values.view(B*n_tiles, C, H, W).to('cuda', dtype=torch.float16)
        with torch.no_grad():
            out  = vision_tower(pv_flat, output_hidden_states=True)
            feat = out.hidden_states[-2][:, 1:, :]   # drop CLS  (B*tiles, patches, D)
        return feat.detach()   # explicit detach — projector owns the graph

    def get_proj_features(pixel_values):
        # Projector — gradients flow here during training
        feat = get_clip_features(pixel_values)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            return projector(feat)

    def get_proj_features_nograd(pixel_values):
        # Validation only — no gradients anywhere
        feat = get_clip_features(pixel_values)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                return projector(feat)

    def align_loss(projected):
        # VICReg variance-covariance loss — no positive pairs needed
        # projected: (N, patches, D)
        fm = projected.mean(dim=1).float()   # (N, D) — fp32 for numerical stability
        N, D = fm.shape
        if N < 2:
            return torch.tensor(0.0, device=fm.device, requires_grad=True)
        fm = fm - fm.mean(dim=0, keepdim=True)

        # Variance: push std of each dim toward 1
        std      = torch.sqrt(fm.var(dim=0, unbiased=False) + 1e-4)
        var_loss = torch.mean(torch.relu(1.0 - std))

        # Covariance: decorrelate dims
        cov      = (fm.T @ fm) / max(N - 1, 1)
        eye      = torch.eye(D, device=cov.device, dtype=cov.dtype)
        cov_loss = ((cov ** 2) * (1 - eye)).sum() / D

        return var_loss + 0.04 * cov_loss

    def run_val_loss(loader, max_batches=20):
        projector.eval()
        losses = []
        for i, b in enumerate(loader):
            if i >= max_batches: break
            pv   = b['pixel_values'].to('cuda')
            feat = get_proj_features_nograd(pv)
            loss = align_loss(feat)
            v    = loss.item()
            if not (v != v) and abs(v) < 1e6:   # skip nan/inf
                losses.append(v)
        projector.train()
        return float(np.mean(losses)) if losses else float('nan')

    class Stage1Dataset(Dataset):
        def __init__(self, df, cache=True):
            self.df    = df.drop_duplicates(subset='image_path').reset_index(drop=True)
            self.cache = {}
            print(f"Stage1Dataset: {len(self.df):,} images", end='')
            if cache:
                print(" - caching...", end='', flush=True)
                for idx, row in self.df.iterrows():
                    try:
                        img = Image.open(str(row['image_path'])).convert('RGB')
                        self.cache[idx] = img.resize((IMG_SIZE, IMG_SIZE))
                    except:
                        self.cache[idx] = Image.new('RGB',(IMG_SIZE,IMG_SIZE),(128,128,128))
                print(" done")
            else: print()
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            return {'image': self.cache.get(idx,
                Image.new('RGB',(IMG_SIZE,IMG_SIZE),(128,128,128)))}

    def s1_collate(batch):
        pv, sz = encode_images([b['image'] for b in batch])
        return {'pixel_values': pv, 'image_sizes': sz}

    uniq = train_df.drop_duplicates(subset='image_path')
    if S1_SUBSAMPLE is None:
        s1_df = uniq.reset_index(drop=True)
        print(f"Stage I: ALL {len(s1_df):,} unique images")
    elif 'disease_label' in uniq.columns:
        nc = uniq['disease_label'].nunique(); per_cls = max(1, S1_SUBSAMPLE//nc)
        s1_df = (uniq.groupby('disease_label', group_keys=False)
                     .apply(lambda g: g.sample(min(len(g),per_cls), random_state=RANDOM_SEED))
                     .reset_index(drop=True))
    else:
        s1_df = uniq.sample(min(len(uniq),S1_SUBSAMPLE), random_state=RANDOM_SEED).reset_index(drop=True)

    val_uniq = val_df.drop_duplicates(subset='image_path').sample(
        min(200, len(val_df)), random_state=RANDOM_SEED).reset_index(drop=True)

    N_W      = min(8, os.cpu_count() or 4)
    s1_train = Stage1Dataset(s1_df,    cache=True)
    s1_val   = Stage1Dataset(val_uniq, cache=True)
    s1_tr_dl = DataLoader(s1_train, batch_size=8, shuffle=True,  collate_fn=s1_collate,
                          num_workers=N_W, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
    s1_vl_dl = DataLoader(s1_val,   batch_size=8, shuffle=False, collate_fn=s1_collate,
                          num_workers=N_W, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4)
    print(f"Train: {len(s1_tr_dl):,} batches | {S1_EPOCHS} epochs")

    steps = (len(s1_tr_dl) // S1_GRAD_ACCUM) * S1_EPOCHS
    opt   = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=S1_LR, weight_decay=1e-4)
    sched = get_linear_schedule_with_warmup(opt, int(steps*0.03), steps)

    for epoch in range(S1_EPOCHS):
        projector.train(); opt.zero_grad(); losses=[]; step=0
        pbar = tqdm(s1_tr_dl, desc=f"S1 E{epoch+1}/{S1_EPOCHS}")
        for i, batch in enumerate(pbar):
            pv   = batch['pixel_values'].to('cuda')
            feat = get_proj_features(pv)
            loss = align_loss(feat)
            v    = loss.item()
            if v != v or abs(v) > 1e6:
                opt.zero_grad(); continue
            (loss / S1_GRAD_ACCUM).backward()
            losses.append(v)
            if (i+1) % S1_GRAD_ACCUM == 0 or (i+1) == len(s1_tr_dl):
                gn = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                opt.step(); sched.step(); opt.zero_grad(); step += 1
                pbar.set_postfix({'loss':  f'{np.mean(losses[-20:]):.4f}',
                                  'gnorm': f'{float(gn):.2f}',
                                  'step':  step})

        tr = float(np.mean(losses)) if losses else float('nan')
        vl = run_val_loss(s1_vl_dl)
        s1_log.append({'epoch': epoch+1, 'train': tr, 'val': vl})
        print(f"S1 E{epoch+1} — train={tr:.4f}  val={vl:.4f}")

        # Save if best (or save anyway if val still nan to preserve weights)
        save_ok = not (vl != vl) and vl < s1_best
        if save_ok or s1_ckpt is None:
            if save_ok: s1_best = vl
            ckpt = CHECKPOINT_DIR / 'stage1_best'
            ckpt.mkdir(parents=True, exist_ok=True)
            torch.save(projector.state_dict(), ckpt / 'projector.pt')
            json.dump({'val_loss': vl if not (vl!=vl) else 999.0,
                       'epoch': epoch+1},
                      open(ckpt/'meta.json','w'))
            s1_ckpt = ckpt
            tag = "best" if save_ok else "saved (val=nan)"
            print(f"  -> {tag}: {ckpt}")

    del opt, sched; free_memory()
    for p in model.parameters(): p.requires_grad = True
    print(f"\nStage I done. Best val={s1_best:.4f}  Checkpoint: {s1_ckpt}")


In [ ]:
# Cell 6: Stage II Datasets + DataLoaders
class Stage2Dataset(Dataset):
    def __init__(self, df, alpha_mode=ALPHA_MODE):
        self.df = (df[df['question_type']!='conversation'].reset_index(drop=True)
                   if 'question_type' in df.columns else df.reset_index(drop=True))
        self.alpha_mode = alpha_mode
        print(f"Stage2Dataset: {len(self.df):,} QA pairs")
    def __len__(self): return len(self.df)
    def _load(self, path):
        try:
            img = Image.open(str(path)); img.verify()
            return Image.open(str(path)).convert('RGB')
        except: return Image.new('RGB',(IMG_SIZE,IMG_SIZE),(128,128,128))
    def _blend(self, img, bbox):
        arr = np.array(img); x1,y1,x2,y2 = [int(v) for v in bbox]; ov = arr.copy()
        cv2.rectangle(ov, (x1,y1), (x2,y2), (255,0,0), 3)
        alpha = random.uniform(96/255, 1.0) if self.alpha_mode=='dynamic' else 200/255
        return Image.fromarray(cv2.addWeighted(ov,alpha,arr,1-alpha,0)).resize((IMG_SIZE,IMG_SIZE))
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]; bbox = _parse_bbox(row['bbox'])
        return {'image':      self._blend(self._load(row['image_path']), bbox),
                'bbox_str':   row.get('bbox_str', f"[{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}]"),
                'question':   str(row['question']),
                'answer':     str(row['answer']),
                'question_type': str(row.get('question_type','open'))}

class ConvDataset(Dataset):
    def __init__(self, df, alpha_mode=ALPHA_MODE):
        self.df = (df[df['question_type']=='conversation'].reset_index(drop=True)
                   if 'question_type' in df.columns else df.iloc[0:0].reset_index(drop=True))
        self.alpha_mode = alpha_mode
        print(f"ConvDataset: {len(self.df):,} turns")
    def __len__(self): return len(self.df)
    def _load(self, path):
        try:
            img = Image.open(str(path)); img.verify()
            return Image.open(str(path)).convert('RGB')
        except: return Image.new('RGB',(IMG_SIZE,IMG_SIZE),(128,128,128))
    def _blend(self, img, bbox):
        arr = np.array(img); x1,y1,x2,y2 = [int(v) for v in bbox]; ov = arr.copy()
        cv2.rectangle(ov, (x1,y1), (x2,y2), (255,0,0), 3)
        alpha = random.uniform(96/255, 1.0) if self.alpha_mode=='dynamic' else 200/255
        return Image.fromarray(cv2.addWeighted(ov,alpha,arr,1-alpha,0)).resize((IMG_SIZE,IMG_SIZE))
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]; bbox = _parse_bbox(row['bbox'])
        q_raw = str(row['question'])
        if '[CONV_TURN_' in q_raw:
            parts = q_raw.split('|'); ctx = parts[0]
            for m in ['[CONV_TURN_1]','[CONV_TURN_2]','[CONV_TURN_3]','Previous context:']:
                ctx = ctx.replace(m,'').strip()
            q = parts[-1].strip() if len(parts)>1 else q_raw
        else: ctx=''; q=q_raw
        return {'image':    self._blend(self._load(row['image_path']), bbox),
                'bbox_str': row.get('bbox_str', f"[{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}]"),
                'question': q, 'context': ctx, 'answer': str(row['answer'])}

def stage2_collate(batch):
    images = [b['image'] for b in batch]; pv, sz = encode_images(images)
    all_ids, all_masks, all_labels = [], [], []
    for b_i, b in enumerate(batch):
        text  = (f"[INST] <image>\nRegion of interest: {b['bbox_str']}\n"
                 f"{b['question']} [/INST] {b['answer']}")
        ids, mask = build_input_ids(text, sz[b_i].tolist())
        labels    = _mask_after_inst(ids)
        all_ids.append(ids); all_masks.append(mask); all_labels.append(labels)
    return dict(pixel_values=pv, image_sizes=sz,
                input_ids=torch.tensor(all_ids, dtype=torch.long),
                attention_mask=torch.tensor(all_masks, dtype=torch.long),
                labels=torch.tensor(all_labels, dtype=torch.long))

def conv_collate(batch):
    images = [b['image'] for b in batch]; pv, sz = encode_images(images)
    all_ids, all_masks, all_labels = [], [], []
    for b_i, b in enumerate(batch):
        ctx_str = f"Previous context: {b['context'][:100]}\n" if b['context'] else ""
        text    = (f"[INST] <image>\nRegion of interest: {b['bbox_str']}\n"
                   f"{ctx_str}{b['question']} [/INST] {b['answer']}")
        ids, mask = build_input_ids(text, sz[b_i].tolist())
        labels    = _mask_after_inst(ids)
        all_ids.append(ids); all_masks.append(mask); all_labels.append(labels)
    return dict(pixel_values=pv, image_sizes=sz,
                input_ids=torch.tensor(all_ids, dtype=torch.long),
                attention_mask=torch.tensor(all_masks, dtype=torch.long),
                labels=torch.tensor(all_labels, dtype=torch.long))

s2_train   = Stage2Dataset(train_df)
s2_val     = Stage2Dataset(val_df)
has_conv   = ('question_type' in train_df.columns
              and 'conversation' in train_df['question_type'].values)
conv_train = ConvDataset(train_df) if has_conv else None

# num_workers=4 (reduced from 8) — 3 DataLoaders × 4 workers = 12 processes
# avoids excessive NFS file handles on shared DGX storage
DL_KW    = dict(num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
s2_tr_dl = DataLoader(s2_train, batch_size=STAGE2_BATCH, shuffle=True,
                       collate_fn=stage2_collate, **DL_KW)
s2_vl_dl = DataLoader(s2_val,   batch_size=STAGE2_BATCH, shuffle=False,
                       collate_fn=stage2_collate, **DL_KW)
conv_dl  = (DataLoader(conv_train, batch_size=STAGE2_BATCH, shuffle=True,
                        collate_fn=conv_collate, **DL_KW)
            if conv_train and len(conv_train) > 0 else None)

print(f"S2: {len(s2_tr_dl):,} train | {len(s2_vl_dl):,} val batches")
b     = next(iter(s2_tr_dl))
n_img = (b['input_ids'] == IMAGE_TOKEN_ID).sum().item()
n_ans = (b['labels'] != -100).sum().item()
print(f"Sanity: image_tokens={n_img} (expect {TOKENS_PER_PATCH*3}), answer_tokens={n_ans}")
assert n_img > 0, f"No image tokens — TOKENS_PER_PATCH wrong ({TOKENS_PER_PATCH})"
assert n_ans > 0, "No answer tokens — check collate"


In [ ]:
# Cell 7: Stage II — Apply LoRA + Train (no unsloth, pure peft)
free_memory()

from peft import get_peft_model, LoraConfig, TaskType, PeftModel

# ── RESUME CHECK (before LoRA init) ──────────────────────────────────────────
# If resuming, load checkpoint adapters directly — skip fresh LoRA init.
# If not resuming, apply fresh LoRA. This avoids wasting compute on a LoRA
# init that would immediately be overwritten by resume load.
s2_log, s2_best, s2_ckpt, patience, start_epoch = [], float('inf'), None, 0, 0

if RESUME_FROM and Path(RESUME_FROM).exists():
    print(f"Resuming from: {RESUME_FROM}")
    model = PeftModel.from_pretrained(model, str(RESUME_FROM))
    meta_p = Path(RESUME_FROM) / 'meta.json'
    if meta_p.exists():
        meta        = json.load(open(meta_p))
        s2_best     = meta.get('val_loss', float('inf'))
        start_epoch = meta.get('epoch', 0)
        print(f"  epoch={start_epoch}  best_val={s2_best:.4f}")
    print("LoRA adapters loaded from checkpoint (skipping fresh LoRA init).")
else:
    # Fresh LoRA init — target language attention layers only
    tgt = []
    for name, _ in model.named_modules():
        if any(t in name for t in ["q_proj","k_proj","v_proj","o_proj"]):
            if any(s in name for s in ["language_model","llm","mistral","text_model"]):
                base = name.rsplit(".",1)[-1]
                if base not in tgt: tgt.append(base)
    if not tgt:
        tgt = ["q_proj","k_proj","v_proj","o_proj"]
    print(f"LoRA targets: {tgt}")

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=tgt,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

# ── Load Stage I projector weights if available ───────────────────────────────
if s1_ckpt and (Path(str(s1_ckpt))/'projector.pt').exists():
    try:
        pj = get_projector(model)
        pj.load_state_dict(
            torch.load(
                Path(str(s1_ckpt)) / 'projector.pt',
                map_location='cuda',
                weights_only=True   # projector.pt is state_dict only — safe
            )
        )
        print(f"Stage I projector loaded from {s1_ckpt}")
    except Exception as e:
        print(f"Stage I projector load warning: {e}")
else:
    print("No Stage I projector — using pretrained LLaVA-v1.6 projector")

model.train()

for i in range(torch.cuda.device_count()):
    used    = torch.cuda.memory_allocated(i) / 1e9
    free_gb = (torch.cuda.get_device_properties(i).total_memory
               - torch.cuda.memory_allocated(i)) / 1e9
    print(f"GPU {i} after LoRA: {used:.2f} GB used | {free_gb:.2f} GB free")

# ── Training utilities ────────────────────────────────────────────────────────
_BATCH_KEYS = ('pixel_values','image_sizes','input_ids','attention_mask','labels')

def val_loss_fn(model, loader, max_batches=30):
    model.eval(); losses = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches: break
            batch = {k: v.to('cuda', non_blocking=True) for k,v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                out = model(**{k:v for k,v in batch.items() if k in _BATCH_KEYS})
            if out.loss is not None: losses.append(out.loss.item())
    model.train()
    return float(np.mean(losses)) if losses else float('nan')

def save_ckpt_s2(model, tag, v_loss, epoch=None):
    path = CHECKPOINT_DIR / tag
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    processor.save_pretrained(path)
    meta = {'val_loss': v_loss, 'tag': tag}
    if epoch is not None: meta['epoch'] = epoch
    json.dump(meta, open(path/'meta.json','w'))
    print(f"  Saved: {path}")
    return path

def train_epoch_s2(model, loader, optimizer, scheduler, grad_accum, label):
    model.train(); optimizer.zero_grad(); losses=[]; step=0
    pbar = tqdm(loader, desc=label)
    for i, batch in enumerate(pbar):
        batch = {k: v.to('cuda', non_blocking=True) for k,v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            out = model(**{k:v for k,v in batch.items() if k in _BATCH_KEYS})
        loss = out.loss
        if loss is None: continue
        (loss / grad_accum).backward(); losses.append(loss.item())
        if (i+1) % grad_accum == 0 or (i+1) == len(loader):
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(); step += 1
            pbar.set_postfix({'loss':  f'{np.mean(losses[-20:]):.4f}',
                              'gnorm': f'{float(gn):.2f}',
                              'step':  step})
    return float(np.mean(losses)) if losses else float('nan')

# ── Train ─────────────────────────────────────────────────────────────────────
s2_steps = (len(s2_tr_dl) // STAGE2_GRAD_ACCUM) * STAGE2_EPOCHS
s2_opt   = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=STAGE2_LR, weight_decay=0.)
s2_sched = get_linear_schedule_with_warmup(
    s2_opt, int(s2_steps * STAGE2_WARMUP), s2_steps)
print(f"Stage II: {STAGE2_EPOCHS} epochs | {s2_steps} steps | "
      f"{len(s2_tr_dl):,} batches/epoch")

for epoch in range(start_epoch, STAGE2_EPOCHS):
    lbl = f"S2 Epoch {epoch+1}/{STAGE2_EPOCHS}"
    tr  = train_epoch_s2(model, s2_tr_dl, s2_opt, s2_sched, STAGE2_GRAD_ACCUM, lbl)
    if conv_dl is not None:
        tr_c = train_epoch_s2(model, conv_dl, s2_opt, s2_sched,
                               STAGE2_GRAD_ACCUM, f"{lbl}[conv]")
        tr   = (tr + tr_c) / 2
    vl = val_loss_fn(model, s2_vl_dl)
    s2_log.append({'epoch': epoch+1, 'train': tr, 'val': vl, 'epoch_end': True})
    print(f"{lbl} — train={tr:.4f}  val={vl:.4f}")
    if vl < s2_best:
        s2_best = vl; patience = 0
        s2_ckpt = save_ckpt_s2(model, 'rllava_stage2_best', vl, epoch=epoch+1)
        print(f"  -> best (val={vl:.4f})")
    else:
        patience += 1
        print(f"  no improvement ({patience}/{EARLY_STOP_PATIENCE})")
        if patience >= EARLY_STOP_PATIENCE:
            print("Early stopping."); break
    save_ckpt_s2(model, 'rllava_stage2_last', vl, epoch=epoch+1)
    free_memory()

print(f"\nStage II done. Best val={s2_best:.4f}")
del s2_opt, s2_sched; free_memory()

import matplotlib.pyplot as plt
with open(WORK_DIR/'training_log.json','w') as f:
    json.dump({'stage1': s1_log, 'stage2': s2_log,
               's1_best': s1_best, 's2_best': s2_best}, f, indent=2)
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for ax, data, title in [
    (axes[0], s1_log, 'Stage I - Projector'),
    (axes[1], [e for e in s2_log if e.get('epoch_end')], 'Stage II - LoRA')]:
    if not data: ax.set_title(title+' (skipped)'); continue
    ep = [e['epoch'] for e in data]
    ax.plot(ep, [e['train'] for e in data], 'o-', label='Train', color='#2563eb')
    ax.plot(ep, [e['val']   for e in data], 's--', label='Val',   color='#f97316')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR/'training_curve.png', dpi=150); plt.show()
print(f"Best checkpoint: {s2_ckpt}")


In [ ]:
# Cell 8: Evaluation — BLEU-4, ROUGE-L, METEOR, Exact Match
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as _meteor
from rouge_score import rouge_scorer as rs
for c in ['punkt','punkt_tab','wordnet']:
    try: nltk.download(c, quiet=True)
    except: pass

_rouge  = rs.RougeScorer(['rougeL'], use_stemmer=True)
_smooth = SmoothingFunction().method1

def bleu4(r,h):
    w = h.lower().split()
    return sentence_bleu([r.lower().split()], w,
                         weights=(.25,)*4, smoothing_function=_smooth) if w else 0.0
def rougel(r,h): return _rouge.score(r,h)['rougeL'].fmeasure
def meteor(r,h):
    try: return _meteor([r.lower().split()], h.lower().split())
    except: return 0.0
def em(r,h): return float(r.strip().lower() == h.strip().lower())

print("Loading best checkpoint for evaluation...")
del model; free_memory()

from transformers import LlavaNextForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

bnb_eval = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
# local_files_only=True — avoids HF re-download on DGX
# (model already cached from Cell 3; set False if running fresh)
_base = LlavaNextForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_eval,
    device_map="auto", low_cpu_mem_usage=True,
    local_files_only=True)
ev = PeftModel.from_pretrained(_base, str(s2_ckpt))
ev.eval()

N_EVAL = min(500, len(s2_val)); preds = []
for idx in tqdm(range(N_EVAL), desc="Eval"):
    sample = s2_val[idx]
    prompt = (f"[INST] <image>\nRegion of interest: {sample['bbox_str']}\n"
              f"{sample['question']} [/INST]")
    inp = processor(text=prompt, images=sample['image'], return_tensors='pt')
    inp = {k: v.to('cuda') for k,v in inp.items()}
    with torch.no_grad():
        out = ev.generate(**inp, max_new_tokens=STAGE2_MAX_NEW_TOK, do_sample=False)
    pred = text_tok.decode(
        out[0, inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    preds.append({'pred': pred, 'gt': sample['answer'],
                  'qt': sample['question_type']})
    del inp, out
    if idx % 100 == 0: free_memory()
del ev; free_memory()

import pandas as pd
df_m = pd.DataFrame([{
    'qt': p['qt'],
    'b4': bleu4(p['gt'], p['pred']),
    'rl': rougel(p['gt'], p['pred']),
    'mt': meteor(p['gt'], p['pred']),
    'em': em(p['gt'], p['pred']),
} for p in preds])

print('\n' + '='*65)
print(f"{'Type':<22} {'BLEU-4':>8} {'ROUGE-L':>8} {'METEOR':>8} {'EM%':>7}  N")
print('-'*65)
for qt in df_m.qt.unique():
    s = df_m[df_m.qt==qt]
    print(f"  {qt:<20} {s.b4.mean():>8.4f} {s.rl.mean():>8.4f} "
          f"{s.mt.mean():>8.4f} {s.em.mean()*100:>6.1f}%  {len(s)}")
print('-'*65)
print(f"  {'OVERALL':<20} {df_m.b4.mean():>8.4f} {df_m.rl.mean():>8.4f} "
      f"{df_m.mt.mean():>8.4f} {df_m.em.mean()*100:>6.1f}%  {len(df_m)}")
print('='*65)

summary = {
    'n_eval': len(df_m), 'checkpoint': str(s2_ckpt),
    'overall': {
        'bleu4':       round(float(df_m.b4.mean()), 4),
        'rouge_l':     round(float(df_m.rl.mean()), 4),
        'meteor':      round(float(df_m.mt.mean()), 4),
        'exact_match': round(float(df_m.em.mean()), 4),
    },
    'by_question_type': {
        qt: {
            'bleu4':       round(float(df_m[df_m.qt==qt].b4.mean()), 4),
            'rouge_l':     round(float(df_m[df_m.qt==qt].rl.mean()), 4),
            'exact_match': round(float(df_m[df_m.qt==qt].em.mean()), 4),
            'n':           int((df_m.qt==qt).sum()),
        } for qt in df_m.qt.unique()
    }
}
with open(WORK_DIR/'eval_metrics.json','w') as f: json.dump(summary, f, indent=2)
with open(WORK_DIR/'val_predictions.json','w') as f:
    json.dump([{'prediction': p['pred'], 'ground_truth': p['gt'],
                'question_type': p['qt']} for p in preds], f, indent=2)

print(f"\nSaved to {WORK_DIR}:")
print(f"  training_curve.png    training_log.json")
print(f"  eval_metrics.json     val_predictions.json")
print(f"  rllava_checkpoints/rllava_stage2_best/  <- final model")
